<a href="https://colab.research.google.com/github/vinodrajs001/agentcore001/blob/main/event_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://youtu.be/bu2cD1pCFTs

In [ ]:
%%writefile requirements.txt
boto3
pandas
requests
ipython
botocore
strands-agents
bedrock-agentcore
bedrock-agentcore-starter-toolkit

In [ ]:
%pip install -qUr requirements.txt

In [ ]:
# Import the userdata module
# DO NOT RUN IF LOCAL
from google.colab import userdata
import os

# Retrieve AWS credentials from Colab Secrets
aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY')
aws_region_name = userdata.get('AWS_REGION_NAME')


# Set the AWS environment variables
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key
os.environ['AWS_REGION_NAME'] = aws_region_name

print("AWS credentials loaded and configured from Colab Secrets.")


In [ ]:
import os
import json
import uuid
import time
import boto3
import logging
import pandas as pd
import requests
from io import StringIO
from datetime import datetime
# from utils import create_agentcore_role
from IPython.display import Markdown,display

from botocore.exceptions import ClientError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger("awssome-event-agent")


REGION = os.getenv('AWS_REGION', 'us-east-1')
UNIQUE_ID = str(uuid.uuid4())[:8]
# MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_ID = "us.amazon.nova-micro-v1:0"
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"


print(f"REGION: {REGION}")
print(f"UNIQUE_ID: {UNIQUE_ID}")
print(f"MODEL_ID: {MODEL_ID}")
print(f"EMBEDDING_MODEL: {EMBEDDING_MODEL}")


In [ ]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(model_id=MODEL_ID)

simple_agent = Agent(
    model=model,
    system_prompt="You are an intelligent event assistant"
)

print("\n Simple Agent Created")

print("\n Testing simle agent \n")
response = simple_agent("which sessions is good to learbout about security with AI Agents, provide answer in 2 -3 lines")


In [ ]:
# Create Knowledge Base
bedrock_agent_client = boto3.client(service_name='bedrock-agent', region_name=REGION)
KB_NAME = f"EmployeeDetailsKB_{UNIQUE_ID}"

kb_response = bedrock_agent_client.create_knowledge_base(
    name=KB_NAME,
    description="Knowledge base for employee details",
    roleArn="arn:aws:iam::610755329309:role/BedrockKBRole",
    knowledgeBaseConfiguration={
        'type': 'VECTOR',
        'vectorKnowledgeBaseConfiguration':{
            'embeddingModelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-embed-text-v2:0',
            'embeddingModelConfiguration':{
                'bedrockEmbeddingModelConfiguration':{
                    'dimensions': 256,
                    'embeddingDataType': 'FLOAT32'
                }
            }
        }
    },
    storageConfiguration={
        'type': 'S3_VECTORS',
        's3VectorsConfiguration':{
            'indexArn': 'arn:aws:s3vectors:us-east-1:610755329309:bucket/bedrock-kb-001/index/employee-details-index'
        }
    }
)

KB_ID = kb_response['knowledgeBase']['knowledgeBaseId']
print(f"KB_ID: {KB_ID}")

In [ ]:
kb = bedrock_agent_client.get_knowledge_base(
    knowledgeBaseId=KB_ID
)

print(kb["knowledgeBase"]["status"])

ds = bedrock_agent_client.get_data_source(
    knowledgeBaseId=KB_ID,
    dataSourceId=data_source_id
)

print(ds["dataSource"]["status"])

In [ ]:
ds_response= bedrock_agent_client.create_data_source(
    knowledgeBaseId=KB_ID,
    name=f"EmployeeDetailsDataSource_{UNIQUE_ID}",
    description="Data source for employee details",
    dataSourceConfiguration={
        'type': 'S3',
        's3Configuration': {
            'bucketArn': 'arn:aws:s3:::bedrock-agentcore-kb-610755329309-us-east-1-an'}},
    vectorIngestionConfiguration={
        'chunkingConfiguration':{
            'chunkingStrategy':'FIXED_SIZE',
            'fixedSizeChunkingConfiguration':{
                'maxTokens': 100,
                'overlapPercentage':10
            }
        }
    }
)

data_source_id=ds_response['dataSource']['dataSourceId']
print(f"Data source created: {data_source_id}")


ingestion_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=data_source_id
)


injection_job_id = ingestion_response['ingestionJob']['ingestionJobId']
print(f"Ingestion job started: {injection_job_id}")

while True:
    status = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=data_source_id,
        ingestionJobId=injection_job_id
    )

    job_status = status["ingestionJob"]["status"]
    print(job_status)

    if job_status in ["COMPLETE", "FAILED"]:
        break

    time.sleep(10)




In [ ]:
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime',region_name=REGION)

test_query = "List all employees who is on leave from 1st June 2026"

print(f"Test Query: {test_query}")

response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={'text': test_query},
    retrievalConfiguration={
        'vectorSearchConfiguration': {
            'numberOfResults': 3
        }
    }
)

print(f"Found {len(response['retrievalResults'])} results:\n")
for idx, result in enumerate[any](response['retrievalResults'],1):
    content = result['content']['text']
    score = result['score']
    print(f"{idx}. Score: {score:.3f}")
    print(f" {content[:200]}...\n")



In [ ]:
# KB search tool
from strands import tool
from typing import List, Dict, Any

@tool
def search_reinvent_sessions(query: str, max_results: int = 3) -> List[Dict[str, Any]]:
    """Search for employee details from knowledge based using semantic search.
    Use this tool when user ask for employee details.

    Args :
      query: The search query.
      max_results: The maximum number of results to return.

    Returns:
      A list of employee details.

    """

    try:
        logger.info(f"Searching KB with query :{query}")

        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': min(max_results,10)
                }
            }
        )

        results = []
        for idx, item in enumerate[Any](response.get('retrievalResults',[]),1):
            result = {
                'rank': idx,
                'content': item.get('content',{}).get('text',''),
                'score': item.get('score',0.0)
            }
            results.append(result)
        logger.info(f"Found {len(results)} RIV sessions")
        return results
    except Exception as e:
        logger.error(f"Error searching KB: {e}")
        return [{"error": f"Failed to search KB : {str(e)}"}]
print("\n KB Search Tool Created")

In [ ]:
# Create agentcore memory

from bedrock_agentcore.memory.constants import StrategyType, ConversationalMessage , MessageRole
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.session import MemorySessionManager

memory_manager = MemoryManager(region_name=REGION)

memory = memory_manager.get_or_create_memory(
    name=f"EventAgentMemory_{UNIQUE_ID}",
    strategies=[
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "UserPreferences",
                "namespaces": ["/users/{actorId}/preferences"],
                "description": "Captures customer preferences and behavior"
            }
        }
    ],
    description="Memory for Employee Agent with user preferences",
    event_expiry_days=30
)

MEMORY_ID = memory.get("id")

print(f"Memory ID: {MEMORY_ID}")



In [ ]:
from botocore import context
from strands.hooks import (
    HookProvider,
    HookRegistry,
    AgentInitializedEvent,
    MessageAddedEvent
)

class MemoryHookProvider(HookProvider):
    # https://youtu.be/bu2cD1pCFTs?t=2198
    def __init__(self):
        logger.info("Initializing Memory Hook Provider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID,REGION)


    def on_agent_initialized(self, event: AgentInitializedEvent):
        logger.info("Agent Initialized")

        actor_id = event.agent.state.get("actor_id")

        if not actor_id:
            logger.warning("Missing actor_id")
            return
        try:
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                query="what are the user's preferences",
                top_k=5
            )

            if preferences:
                logger.info(f"User preferences: {preferences}")
                pref_messages = []
                for pref in preferences:
                    pref_text = pref.get('content',{}).get('text','')
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)
                            pref_messages.append(f"- {pref_json.get('preferences', pref_text)}")
                        except:
                            pref_messages.append(f"{pref_text}\n")
                if pref_messages:
                    context = "\n".join(pref_messages)
                    event.agent.system_prompt += f"**User Preferences {context}"
                    logger.info("Addeded user preference")
            else:
                logger.info("No user preferences found")
        except Exception as e:
            logger.error(f"Error retrieving user preferences: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        logger.info("Message Added")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")

        if not actor_id or not session_id:
            logger.warning("Missing actor_id or session_id")
            return

        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT

            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[
                    ConversationalMessage(
                      message_content, message_role
                    )
                ]
            )
            logger.info("Added message to memory")
        except Exception as e:
            logger.error(f"Error adding message to memory: {e}")

    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering Memory Hooks")
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

print("\n Memory Hook Provider Created")


In [ ]:
from botocore import context
from strands.hooks import (
    HookProvider,
    HookRegistry,
    AgentInitializedEvent,
    MessageAddedEvent
)

class MemoryHookProvider(HookProvider):
    # https://youtu.be/bu2cD1pCFTs?t=2198
    def __init__(self):
        logger.info("Initializing Memory Hook Provider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID,REGION)


    def on_agent_initialized(self, event: AgentInitializedEvent):
        logger.info("Agent Initialized")

        actor_id = event.agent.state.get("actor_id")

        if not actor_id:
            logger.warning("Missing actor_id")
            return
        try:
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                query="what are the user's preferences",
                top_k=5
            )

            if preferences:
                logger.info(f"User preferences: {preferences}")
                pref_messages = []
                for pref in preferences:
                    pref_text = pref.get('content',{}).get('text','')
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)
                            pref_messages.append(f"- {pref_json.get('preferences', pref_text)}")
                        except:
                            pref_messages.append(f"{pref_text}\n")
                if pref_messages:
                    context = "\n".join(pref_messages)
                    event.agent.system_prompt += f"**User Preferences {context}"
                    logger.info("Addeded user preference")
            else:
                logger.info("No user preferences found")
        except Exception as e:
            logger.error(f"Error retrieving user preferences: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        logger.info("Message Added")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")

        if not actor_id or not session_id:
            logger.warning("Missing actor_id or session_id")
            return

        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT

            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[
                    ConversationalMessage(
                      message_content, message_role
                    )
                ]
            )
            logger.info("Added message to memory")
        except Exception as e:
            logger.error(f"Error adding message to memory: {e}")

    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering Memory Hooks")
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

print("\n Memory Hook Provider Created")


In [ ]:
# create agent with memory and RAG

ACTOR_ID = f"user-{UNIQUE_ID}"
session_id_1 = f"session-1-{str(uuid.uuid4())[:8]}"

print(f"ACTOR_ID: {ACTOR_ID}")
print(f"session_id_1: {session_id_1}")
print(f"memorry id: {MEMORY_ID}")

memory_hook = MemoryHookProvider()

# Create agent with memory & RAG

memory_rag_agent = Agent(
    model=model,
    system_prompt="You are an intelligent employee details assistant, you can retrieve emloyee details",
    hooks=[memory_hook],
    tools=[search_reinvent_sessions],
    state={
        "actor_id": ACTOR_ID,
        "session_id": session_id_1
    }
)

print("Memory enabled RAG agent created")



In [ ]:
# share personal information
# https://youtu.be/bu2cD1pCFTs?t=2539
response = memory_rag_agent(
    """Hi! Let me tell you about myself
      - My name is Vinod Shalgar
      - Work as a HR assistant
      - I am mainly looking for employee details such as leave balances, who is on leave
    """
)

In [ ]:
from bedrock_agentcore.memory.models import MemoryRecord
# validate long term memory
# https://youtu.be/bu2cD1pCFTs?t=2573

memory_session_manager = MemorySessionManager(MEMORY_ID,REGION)

try:
    response = memory_session_manager.list_long_term_memory_records(
        namespace_prefix=f"/users/{ACTOR_ID}/preferences",
        max_results=10
    )

    if response:
            print(f"\n Found{len(response)} preferences")
            for idx, pref in enumerate[MemoryRecord](response,1):
                content = pref.get('content',{}).get('text','')
                print(f"{idx}. {content}")
    else:
        print("No preferences found")
except Exception as e:
    print(f"Error retrieving preferences: {e}")
# https://youtu.be/bu2cD1pCFTs?t=2614

In [ ]:
# https://youtu.be/bu2cD1pCFTs?t=2645
# Cross Session Memory

session_id_2 = f"session-2-{str(uuid.uuid4())[:8]}"
print(f"session_id_2: {session_id_2}")
ACTOR_ID = f"user-{UNIQUE_ID}"

new_memory_hook = MemoryHookProvider()
new_memory_rag_agent = Agent(
    model=model,
    hooks=[new_memory_hook],
    tools=[search_reinvent_sessions],
    system_prompt="You are an intelligent employee details assistant, you can retrieve emloyee details",
    state={
        "actor_id": ACTOR_ID, # same user
        "session_id": session_id_2 # different sessions
    }
)
#https://youtu.be/bu2cD1pCFTs?t=2659
response = new_memory_rag_agent("List of employees who are on leave")
display(Markdown(f"**Agent:**{response.message['content'][0]['text']}"))



In [ ]:
# Try with another sessions
session_id_3 = f"session-2-{str(uuid.uuid4())[:8]}"
print(f"session_id_3: {session_id_3}")
ACTOR_ID = f"user-{UNIQUE_ID}"

new_memory_hook = MemoryHookProvider()
new_memory_rag_agent = Agent(
    model=model,
    hooks=[new_memory_hook],
    tools=[search_reinvent_sessions],
    system_prompt="You are an intelligent employee details assistant, you can retrieve emloyee details",
    state={
        "actor_id": ACTOR_ID, # same user
        "session_id": session_id_3 # different sessions
    }
)
#https://youtu.be/bu2cD1pCFTs?t=2659
response = new_memory_rag_agent("Provide me company leave policies")
display(Markdown(f"**Agent:**{response.message['content'][0]['text']}"))


In [ ]:
%%writefile event_agent.py
# AgentCore Runtime / create agent file for runtime
# https://youtu.be/bu2cD1pCFTs?t=2906
import os
import json
import boto3
import logging
from strands import Agent, tool
from typing import Dict, Any , List
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp  # for agentcore
from bedrock_agentcore.memory.session import MemorySessionManager
from strands.hooks import AgentInitializedEvent , HookProvider, HookRegistry , MessageAddedEvent
from bedrock_agentcore.memory.constants import StrategyType , ConversationalMessage, MessageRole

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)

logger = logging.getLogger("event-agent")

app = BedrockAgentCoreApp() # for agentcore

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
REGION = os.getenv('AWS_REGION')
KB_ID = os.getenv('KB_ID')

agent = None
bedrock_agent_runtime = None

class MemoryHookProvider(HookProvider):
    # https://youtu.be/bu2cD1pCFTs?t=2198
    def __init__(self):
        logger.info("Initializing Memory Hook Provider")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID,REGION)


    def on_agent_initialized(self, event: AgentInitializedEvent):
        logger.info("Agent Initialized")

        actor_id = event.agent.state.get("actor_id")

        if not actor_id:
            logger.warning("Missing actor_id")
            return
        try:
            preferences = self.memory_session_manager.search_long_term_memories(
                namespace_prefix=f"/users/{actor_id}/preferences",
                query="what are the user's preferences",
                top_k=5
            )

            if preferences:
                logger.info(f"User preferences: {preferences}")
                pref_messages = []
                for pref in preferences:
                    pref_text = pref.get('content',{}).get('text','')
                    if pref_text:
                        try:
                            pref_json = json.loads(pref_text)
                            pref_messages.append(f"- {pref_json.get('preferences', pref_text)}")
                        except:
                            pref_messages.append(f"{pref_text}\n")
                if pref_messages:
                    context = "\n".join(pref_messages)
                    event.agent.system_prompt += f"**User Preferences {context}"
                    logger.info("Addeded user preference")
            else:
                logger.info("No user preferences found")
        except Exception as e:
            logger.error(f"Error retrieving user preferences: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        logger.info("Message Added")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")

        if not actor_id or not session_id:
            logger.warning("Missing actor_id or session_id")
            return

        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT

            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[
                    ConversationalMessage(
                      message_content, message_role
                    )
                ]
            )
            logger.info("Added message to memory")
        except Exception as e:
            logger.error(f"Error adding message to memory: {e}")

    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering Memory Hooks")
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

print("\n Memory Hook Provider Created")

@tool
def search_reinvent_sessions(query: str, max_results: int = 3) -> List[Dict[str, Any]]:
    """Search for employee details from knowledge based using semantic search.
    Use this tool when user ask for employee details.

    Args :
      query: The search query.
      max_results: The maximum number of results to return.

    Returns:
      A list of employee details.

    """

    try:
        logger.info(f"Searching KB with query :{query}")

        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': min(max_results,10)
                }
            }
        )

        results = []
        for idx, item in enumerate[Any](response.get('retrievalResults',[]),1):
            result = {
                'rank': idx,
                'content': item.get('content',{}).get('text',''),
                'score': item.get('score',0.0)
            }
            results.append(result)
        logger.info(f"Found {len(results)} RIV sessions")
        return results
    except Exception as e:
        logger.error(f"Error searching KB: {e}")
        return [{"error": f"Failed to search KB : {str(e)}"}]
print("\n KB Search Tool Created")


def initialize_agent(actor_id, session_id):
    """Initialize the agent"""
    global agent

    model = BedrockModel(model_id=MODEL_ID)
    memory_hook = MemoryHookProvider()

    agent = Agent(
        model=model,
        hooks=[memory_hook],
        tools=[search_reinvent_sessions],
        system_prompt="""Your are a intelligent assistant that helps users find information
        about employee leaves, leave policy, employee information.
        If you get enough information in the first search don't do additional tool calls.
        Remember user preferences and provide personalized recommendations""",
        state={
            "actor_id": actor_id,
            "session_id": session_id
        }
    )

@app.entrypoint
def runtime_agent(payload, context):
    """Main entry point for the runtime agent"""
    global agent

    user_input = payload.get("prompt")
    actor_id = payload.get("actor_id")
    session_id = contect.session_id

    if not user_input:
        return "Error: Missiong 'prompt' field"

    if agent is None:
        initialize_agent(actor_id, session_id)
    else:
        agent.state["session_id"] = session_id


    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Writing event_agent.py


In [ ]:
# Execution Role
agent_name = f"event_agent_{UNIQUE_ID}"
execution_role_arn = create_agentcore_role(agent_name,region=REGION)

time.sleep(30)

In [ ]:
# configure and launch runtime

from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

print(f"Configuring runtime agent : {agent_name}")

agentcore_runtime.configure(
    entrypoint="event_agent.py",
    execution_role=execution_role_arn["Role"]["Arn"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    non_interactive=True,
    memory_mode="NO_MEMORY"
)

print("Runtime Configured")

agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": MEMORY_ID,
        "KB_ID": KB_ID,
        "MODEL_ID": MODEL_ID,
        "AWS_REGION": REGION
    }
)

print("Runtime Launched")
# test
while True:
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    if status == 'READY':
        print(f"Runtime is ready")
        break
    elif status in ['CREATE_FAILED', 'DELETE_FAILED','UPDATE_FAILED']:
        print(f"Runtime deploy failed {status}")
        break
    print(f"Status {status}")
    time.sleep(10)